In [65]:
%pwd

'/mnt/DATA/Local/Source/Python/semester_9/AIP391/Video_anomaly_detection'

In [66]:
%cd  /home/trong/Downloads/Local/Source/Python/semester_9/AIP391/Video_anomaly_detection/

/mnt/DATA/Local/Source/Python/semester_9/AIP391/Video_anomaly_detection


/home/trong/PyVenv/py311/Video_anomaly_detection/AI/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [67]:
import re
import os
from AI.src.utils.misc import draw_anomaly_graph
from AI.src.data.dataset import VADFrameLevelDataset
from Web.src.be.src.utils.video_utils import find_anomaly_regions


In [68]:
def parse_pred_file(file_path):
    """
    Parse the prediction result file that contains scores and ground truth labels
    
    Returns:
        all_scores: List of prediction score lists
        all_anomaly_ranges: List of lists of anomaly regions (start, end) for each video
        video_indices: List of video indices
    """
    all_scores = []
    all_labels = []
    all_anomaly_ranges = []
    video_indices = []
    
    with open(file_path, 'r') as f:
        content = f.read()
    
    # Find all instances of the pattern [scores],[labels],index
    pattern = r'\[(.*?)\],\[(.*?)\],(\d+)'
    matches = re.findall(pattern, content)
    
    for match in matches:
        scores_str, labels_str, video_idx = match
        
        # Parse scores
        scores = [float(s.strip()) for s in scores_str.split(',')]
        
        # Parse labels
        labels = [int(l.strip()) for l in labels_str.split(',')]
        
        # Find anomaly regions from labels
        anomaly_ranges = []
        start = None
        
        for i, label in enumerate(labels):
            if label == 1 and start is None:
                start = i
            elif label == 0 and start is not None:
                anomaly_ranges.append((start, i - 1))
                start = None
        
        # If there's an anomaly that extends to the end of the video
        if start is not None:
            anomaly_ranges.append((start, len(labels) - 1))
        
        all_scores.append(scores)
        all_labels.append(labels)
        all_anomaly_ranges.append(anomaly_ranges)
        video_indices.append(int(video_idx))
    
    return all_scores, all_anomaly_ranges, video_indices

In [69]:
var_dataset = "UBI-FIGHT"
dataset_root = f"/home/trong/Downloads/Dataset/VAD/tmp_test/final/{var_dataset}/test"  # Adjust to your Windows path
annotation_file = "label.csv"

dataset = VADFrameLevelDataset(
    root = dataset_root,
    annotation=annotation_file,
    loader="v4"
)
video_paths = dataset._VADFrameLevelDataset__annotation["path"].tolist()
video_names = {i: os.path.splitext(os.path.basename(path))[0] for i, path in enumerate(video_paths)}

In [70]:
model_name = "model_3"
video_scores,labels,video_indices = parse_pred_file(f"/home/trong/Downloads/Dataset/VAD/tmp_test/final/{var_dataset}/{model_name}_pred_result.txt")
output_dir = f"/home/trong/Downloads/Dataset/VAD/tmp_test/final/{var_dataset}/plot/{model_name}/fig_1"

os.makedirs(output_dir, exist_ok=True)

output_dir_fig2 = f"/home/trong/Downloads/Dataset/VAD/tmp_test/final/{var_dataset}/plot/{model_name}/fig_2"
os.makedirs(output_dir_fig2, exist_ok=True)

##fig1

In [71]:
for i, video_idx in enumerate(video_indices):
    pred = video_scores[i]
    video_anomaly_ranges = labels[i]
    video_name = video_names.get(video_idx, f"Video_{video_idx}")
    
    # print(f"\nProcessing video: {video_name} (index: {video_idx})")
    save_path = os.path.join(output_dir,f"fig1_{video_name}.png")

    draw_anomaly_graph(preds=pred,
                       anomaly_ranges=video_anomaly_ranges,
                       video_name=video_name,
                       save_path=save_path
                       )

Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_1/fig1_UBI-FIGHT_000000.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_1/fig1_UBI-FIGHT_000000.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_1/fig1_UBI-FIGHT_000003.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_1/fig1_UBI-FIGHT_000003.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_1/fig1_UBI-FIGHT_000004.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_1/fig1_UBI-FIGHT_000004.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_1/fig1_UBI-FIGHT_000000.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_1/fig1_UBI-FIGHT_000000.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test

##fig2

In [72]:
# Thông số cho phát hiện peak
window_length = 15
polyorder = 6
height = 0.7
prominence = 0.3
os.environ.get("MERGE_GAP", 5)


# Tạo thư mục để lưu biểu đồ



for i, video_idx in enumerate(video_indices):
    pred = video_scores[i]
    video_anomaly_ranges = labels[i]
    video_name = video_names.get(video_idx, f"Video_{video_idx}")
    
    # print(f"\nProcessing video: {video_name} (index: {video_idx})")
    
    # 1. Smooth the data
    detected_regions, processed_scores, peaks = find_anomaly_regions(pred,
                                                                     window_length, polyorder,
                                                                     height, prominence,
                                                                     )
    
    # Print peak information
    # print(f"Video {video_name} has {len(peaks)} detected peaks")
    # if len(peaks) > 0:
    #     print(f"Peaks at frames: {peaks}")
    #     print(f"Detected anomaly regions: {detected_regions}")
    
    # 3. Draw graph with ground truth and detected anomaly regions
    save_path = os.path.join(output_dir_fig2, f"fig2_{video_name}.png")
    
    draw_anomaly_graph(
        preds=pred,
        anomaly_ranges=video_anomaly_ranges,
        video_name=video_name,
        save_path=save_path,
        smooth_pred=processed_scores,
        smooth_label="Smoothed pred",
        additional_anomaly_ranges=detected_regions,
        additional_anomaly_color="green"
    )

Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_2/fig2_UBI-FIGHT_000000.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_2/fig2_UBI-FIGHT_000000.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_2/fig2_UBI-FIGHT_000003.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_2/fig2_UBI-FIGHT_000003.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_2/fig2_UBI-FIGHT_000004.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_2/fig2_UBI-FIGHT_000004.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_2/fig2_UBI-FIGHT_000000.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test/final/UBI-FIGHT/plot/model_3/fig_2/fig2_UBI-FIGHT_000000.png
Plot saved to /home/trong/Downloads/Dataset/VAD/tmp_test